In [3]:
!pip install -i https://test.pypi.org/simple/ lowresource-llm-evaluation==0.2.6
!pip install --upgrade sacrebleu
!pip install --upgrade transformers
!pip install python-Levenshtein
!pip install bitsandbytes accelerate

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://test.pypi.org/simple/
INFO: pip is looking at multiple versions of lowresource-llm-evaluation to determine which version is compatible with other requirements. This could take a while.


ERROR: Could not find a version that satisfies the requirement python-Levenshtein (from lowresource-llm-evaluation) (from versions: none)
ERROR: No matching distribution found for python-Levenshtein


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from google.colab import drive
import time
import json
import gc

drive.mount('/content/drive')
base = "drive/MyDrive/EvaluacionTFG/"


load_dotenv(base + "secrets.env")
login()#token=os.getenv("HF_TOKEN"))

In [ ]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def evaluate_benchmark(model_name, idioma, N= 20, debug=False):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=  torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=True,
        tie_word_embeddings=False # Added to silence the warning about tied weights
    )
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    df_textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet").sample(N) , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet").sample(N), 
            "gallego": load_gallego() }
    results = benchmark(model, tokenizer, 
            df_textos = df_textos[idioma],
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").sample(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").sample(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            debug=debug)
    # Lberamos GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Borrar el modelo ha fallado")
        print(e)
    return results

# Aranés

## Mistral 7B 

In [ ]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, 500)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 65.81 MiB is free. Including non-PyTorch memory, this process has 14.50 GiB memory in use. Of the allocated memory 14.30 GiB is allocated by PyTorch, and 69.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Salamandra

In [ ]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma,500)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en -2.978052751223246
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en -0.2748485803604126
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en -1.5580504655838012

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.7278886821488149                                   |
| entropy         | 7.40689048516694                                     |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.4103602030479486                                   |
| freq_comparison | {'es': 0.4504067502863246, 'fr': 0.2857663323530033} |
| calidad         | 0.24541706720334747                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+------------------+
| Clave | Valo

## Gemma

In [ ]:
idioma = "aranes"
modelo = "google/gemma-7b"
resultados = evaluate_benchmark(modelo, idioma, 500)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 